In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import SolverFactory
from pyomo.environ import *

In [2]:
df = pd.read_excel('app20.xlsx')
df.set_index('origem', inplace=True)
df

,loja 3,loja 4,loja 5,loja 6
origem,,,,
loja1,54,17,23,30
loja2,24,18,19,31


In [3]:
restr_maximo = {
    0: 16,
    1: 18
}


demanda = 10

In [4]:
model = pyo.ConcreteModel()

model.origens = pyo.RangeSet(0, len(df)-1)
model.lojas = pyo.Set(initialize=df.columns)

model.x = pyo.Var(model.origens, model.lojas, bounds=(0, None), domain=pyo.NonNegativeIntegers)

def obj(model):
    return sum(model.x[o, l] * df.iloc[o][l] for o in model.origens for l in model.lojas)
model.obj = pyo.Objective(rule=obj, sense=pyo.minimize)

def rest_maximo(model,o):
    return sum(model.x[o, l] for l in model.lojas) <= restr_maximo[o]
model.restricao_maximo = pyo.Constraint(model.origens, rule=rest_maximo)

def minimo_loja(model,l):
    return sum(model.x[o, l] for o in model.origens) >= 5
model.restricao_minimo_loja = pyo.Constraint(model.lojas, rule=minimo_loja)

def rest_demanda(model,l):
    return sum(model.x[o, l] for o in model.origens) <= demanda
model.restricao_demanda = pyo.Constraint(model.lojas, rule=rest_demanda)

In [5]:
model.pprint()

1 Set Declarations
    lojas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {'loja 3', 'loja 4', 'loja 5', 'loja 6'}

1 RangeSet Declarations
    origens : Dimen=1, Size=2, Bounds=(0, 1)
        Key  : Finite : Members
        None :   True :   [0:1]

1 Var Declarations
    x : Size=8, Index=origens*lojas
        Key           : Lower : Value : Upper : Fixed : Stale : Domain
        (0, 'loja 3') :     0 :  None :  None : False :  True : NonNegativeIntegers
        (0, 'loja 4') :     0 :  None :  None : False :  True : NonNegativeIntegers
        (0, 'loja 5') :     0 :  None :  None : False :  True : NonNegativeIntegers
        (0, 'loja 6') :     0 :  None :  None : False :  True : NonNegativeIntegers
        (1, 'loja 3') :     0 :  None :  None : False :  True : NonNegativeIntegers
        (1, 'loja 4') :     0 :  None :  None : False :  True : NonNegativeIntegers
        (1, 'loja 5') :     0 :  None :

In [6]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmpb7_62f00.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmp7a6pqep8.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmp7a6pqep8.pyomo.lp
Objective sense      : Minimize
Variables            :       8  [General Integer: 8]
Objective nonzeros   :       8
Linear constraints   :      10  [Less: 6,  Greater: 4]
  Nonzeros           :      24
  RHS nonzeros       :      10

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 17

In [7]:
for m, v in model.x:
    print(f'Quantidade da loja {m+1} para {v}: {model.x[m, v].value}')
print(f'Custo Total: R$ {model.obj():.2f}')

Quantidade da loja 1 para loja 3: 0.0
Quantidade da loja 1 para loja 4: 5.0
Quantidade da loja 1 para loja 5: 0.0
Quantidade da loja 1 para loja 6: 5.0
Quantidade da loja 2 para loja 3: 5.0
Quantidade da loja 2 para loja 4: -0.0
Quantidade da loja 2 para loja 5: 5.0
Quantidade da loja 2 para loja 6: -0.0
Custo Total: R$ 450.00
